In [7]:
# 필요한 라이브러리 설치
!pip install pandas
!pip install firebase-admin

import pandas as pd
import json
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore
import os

# Firebase 프로젝트의 서비스 계정 키 파일 경로를 설정합니다.
# 이 파일은 Firebase 콘솔에서 다운로드해야 합니다.
# 예시: cred = credentials.Certificate("path/to/your-service-account-key.json")
# 실제 VS Code 환경에서는 다운로드한 JSON 파일 경로를 입력해야 합니다.
# 파일 이름만 입력하려면 현재 작업 디렉토리에 파일을 놓아야 합니다.
# ========== 이 부분을 수정해 주세요! ==========
# 예시: service_account_key_path = "C:/Users/your-username/Downloads/firebase-admin-sdk.json"
service_account_key_path = 'reservationmusicrooom-firebase-adminsdk-fbsvc-d726f4d34d.json'
# ============================================

try:
    if not firebase_admin._apps:
        # 파일이 존재하는지 확인합니다.
        if os.path.exists(service_account_key_path):
            cred = credentials.Certificate(service_account_key_path)
            firebase_admin.initialize_app(cred)
            print("Firebase SDK가 성공적으로 초기화되었습니다.")
        else:
            print(f"오류: 서비스 계정 키 파일 '{service_account_key_path}'을(를) 찾을 수 없습니다.")
            print("Firebase 콘솔에서 다운로드하여 올바른 경로에 두었는지 확인해 주세요.")
            raise FileNotFoundError("Service account key file not found.")

except Exception as e:
    print(f"Firebase 초기화 중 오류가 발생했습니다: {e}")
    raise

# Firestore 클라이언트 초기화
db = firestore.client()
# 앱 이름에 해당하는 appId를 설정합니다.
appId = "wegobe"

# =========================================================================
# 1. Excel 파일 읽기 및 데이터 전처리
# =========================================================================

# 파일 경로를 입력하세요.
# CSV 파일이 아닌 Excel 파일(.xlsx)을 직접 읽도록 수정했습니다.
file_path = "wegobe.xlsx"
try:
    # Excel 파일을 pandas DataFrame으로 읽기 (시트 이름 지정)
    df = pd.read_excel(file_path, sheet_name='Sheet1', header=None)

    # 데이터 전치(Transpose) - 행과 열을 바꿉니다.
    transposed_df = df.T

    # 첫 행(이름, 주소, 사이트 등)을 새로운 헤더로 설정
    transposed_df.columns = transposed_df.iloc[1]
    
    # 실제 데이터만 남기기 (메타데이터 행 제외)
    data_df = transposed_df.iloc[2:]
    
    # 컬럼 이름이 '이름', '주소', '사이트', '예약 가능 시간대'인지 확인합니다.
    expected_cols = ['이름', '주소', '사이트', '예약 가능 시간대']
    if not all(col in data_df.columns for col in expected_cols):
        raise ValueError("파일의 헤더가 예상과 다릅니다. '이름', '주소', '사이트', '예약 가능 시간대' 컬럼이 있는지 확인하세요.")

    # Firestore에 저장할 JSON 형식으로 데이터 변환
    formatted_data = []
    for index, row in data_df.iterrows():
        # '예약 가능 시간대' 열의 JSON 문자열을 처리합니다.
        schedule_str = row.get('예약 가능 시간대', '[]')
        
        # CSV에서 이스케이프된 따옴표(예: "" -> ")를 올바르게 처리
        if isinstance(schedule_str, str):
            schedule_str_cleaned = schedule_str.replace('""', '"')
        else:
            schedule_str_cleaned = '[]'

        try:
            schedule_json = json.loads(schedule_str_cleaned)
        except json.JSONDecodeError as e:
            print(f"JSON 디코딩 실패: {schedule_str_cleaned}, 오류: {e}")
            schedule_json = []

        space = {
            'name': row.get('이름', '이름 없음'),
            'location': row.get('주소', '주소 없음'),
            'googleForm': row.get('사이트', None),
            'phone': row.get('전화', '전화번호 없음'), # 파일에 '전화' 열이 없으므로 임시 값 사용
            'schedule': schedule_json
        }
        formatted_data.append(space)

    print("데이터 전처리 완료. Firestore에 업로드할 데이터:")
    print(json.dumps(formatted_data, indent=2, ensure_ascii=False))

    # =========================================================================
    # 2. Firestore 데이터 업데이트 (기존 데이터 삭제 후 새로운 데이터 추가)
    # =========================================================================
    # Firestore에 저장되는 경로는 'artifacts/{appId}/public/data/{collectionName}'입니다.
    # {appId}에 'wegobe'가 들어갑니다.
    collection_ref = db.collection(f'artifacts/{appId}/public/data/rehearsalSpaces')

    # 기존 데이터 모두 삭제
    docs = collection_ref.stream()

    batch = db.batch()
    for doc_ref in docs:
        batch.delete(doc_ref.reference)
    batch.commit()
    print("기존 데이터 삭제 완료.")

    # 새로운 데이터 추가
    new_batch = db.batch()
    for item in formatted_data:
        new_doc_ref = collection_ref.document()
        new_batch.set(new_doc_ref, item)
    new_batch.commit()
    print("새로운 데이터가 Firestore에 성공적으로 추가되었습니다.")

except FileNotFoundError as e:
    print(f"오류: {e}")
except Exception as e:
    print(f"예상치 못한 오류가 발생했습니다: {e}")


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
데이터 전처리 완료. Firestore에 업로드할 데이터:
[
  {
    "name": "그라운드합주실 본점",
    "location": "서울 마포구 양화로 147 지하2층",
    "googleForm": "https://naver.me/xzxmXyI3",
    "phone": "전화번호 없음",
    "schedule": [
      {
        "day": "매일",
        "time": "오전9시~오전6시"
      }
    ]
  },
  {
    "name": "제시뮤직 합주실 홍대점",
    "location": "서울 마포구 월드컵북로1길 18 지하",
    "googleForm": "https://naver.me/GipBMGRJ",
    "phone": "전화번호 없음",
    "schedule": [
      {
        "day": "매일",
        "time": "오전9시~오전2시"
      }
    ]
  },
  {
    "name": "라디오가가합주실 신촌점",
    "location": "서울 마포구 신촌로16길 10",
    "googleForm": "https://naver.me/FlZ1oEWa",
    "phone": "전화번호 없음",
    "schedule": [
      {
        "day": "매일",
        "time": "오전 10시~오전12시"
      }
    ]
  },
  {
    "name"